# Transaction Data Quality Audit

This notebook performs a structured data quality audit on a synthetic transaction dataset.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [2]:
df = pd.read_csv('customer_transactions_sample.csv')
print('Shape:', df.shape)
df.head(10)

Shape: (120, 9)


,transaction_id,transaction_date,customer_id,product,region,payment_method,status,amount,channel
0,TX00001,2025-01-01,C0018,Laptop,APAC,PayPal,Completed,49.0,Online
1,TX00002,2025-01-02,C0027,Monitor,APAC,Transfer,Completed,388.0,Store
2,TX00003,2025-01-03,C0026,Laptop,EU,PayPal,Completed,329.0,Store
3,TX00004,2025-01-04,NaN,Monitor,APAC,Transfer,Completed,222.0,Online
4,TX00005,2025-01-05,C0034,Monitor,EU,PayPal,Pending,219.0,Online
5,TX00006,2025-01-06,C0023,Laptop,US,Card,Completed,430.0,Online
6,TX00007,2025-01-07,C0004,Headphones,US,PayPal,Completed,47.0,Online
7,TX00008,2025-01-08,C0030,Keyboard,EU,Cash,Completed,350.0,Online
8,TX00009,2025-01-09,C0023,Headphones,EU,Transfer,Completed,NaN,Store
9,TX00010,2025-01-10,C0025,Laptop,APAC,Cash,Completed,51.0,Store


## Dataset overview and business use case

The dataset contains customer transaction records that could be used for revenue reporting, payment analysis, customer behavior analysis, and fraud detection.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    120 non-null    object
 1   transaction_date  120 non-null    object
 2   customer_id       119 non-null    object
 3   product           120 non-null    object
 4   region            120 non-null    object
 5   payment_method    120 non-null    object
 6   status            120 non-null    object
 7   amount            119 non-null    object
 8   channel           119 non-null    object
dtypes: object(9)
memory usage: 8.6+ KB


In [4]:
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'missing': df.isna().sum().values,
    'missing_rate_%': (df.isna().mean() * 100).round(1).values,
    'unique_values': df.nunique(dropna=True).values
})
summary

,column,dtype,missing,missing_rate_%,unique_values
0,transaction_id,object,0,0.0,119
1,transaction_date,object,0,0.0,120
2,customer_id,object,1,0.8,38
3,product,object,0,0.0,6
4,region,object,0,0.0,4
5,payment_method,object,0,0.0,6
6,status,object,0,0.0,5
7,amount,object,1,0.8,110
8,channel,object,1,0.8,2


## Data quality issues

We classify issues using completeness, uniqueness, validity, consistency, and integrity.

In [5]:
issues = []

# Completeness
for col in ['transaction_id', 'transaction_date', 'customer_id', 'payment_method', 'amount', 'channel']:
    miss_idx = df.index[df[col].isna() | (df[col].astype(str).str.strip() == '')].tolist()
    for i in miss_idx:
        issues.append([i+1, col, 'Completeness', 'Medium' if col in ['payment_method', 'channel'] else 'High', 'Missing or blank value'])

# Uniqueness
dup_mask = df.duplicated(subset=['transaction_id'], keep=False)
for i in df.index[dup_mask]:
    issues.append([i+1, 'transaction_id', 'Uniqueness', 'High', 'Duplicate transaction identifier'])

# Validity
amount_num = pd.to_numeric(df['amount'], errors='coerce')
date_parsed = pd.to_datetime(df['transaction_date'], errors='coerce', dayfirst=False)
for i in df.index[(amount_num.isna()) | (amount_num <= 0) | (amount_num > 10000)]:
    issues.append([i+1, 'amount', 'Validity', 'High', 'Invalid amount: non-numeric, non-positive, or extreme'])
for i in df.index[date_parsed.isna()]:
    issues.append([i+1, 'transaction_date', 'Validity', 'High', 'Invalid or unparsable date'])
for i in df.index[~df['region'].astype(str).str.upper().isin(['EU', 'US', 'APAC'])]:
    issues.append([i+1, 'region', 'Validity', 'Medium', 'Invalid region code'])

# Consistency
for i in df.index[df['payment_method'].astype(str).str.lower().isin(['card', 'paypal', 'cash', 'transfer']) & (df['payment_method'] != df['payment_method'].astype(str).str.title())]:
    issues.append([i+1, 'payment_method', 'Consistency', 'Low', 'Inconsistent casing'])
for i in df.index[df['status'].astype(str).str.strip() != df['status'].astype(str).str.strip().str.title()]:
    issues.append([i+1, 'status', 'Consistency', 'Low', 'Inconsistent status formatting'])

# Integrity
valid_dates = pd.to_datetime(df['transaction_date'], errors='coerce')
for i in df.index[valid_dates.isna() == False]:
    pass

issues_df = pd.DataFrame(issues, columns=['affected_row', 'column', 'dimension', 'severity', 'issue'])
issues_df = issues_df.drop_duplicates().sort_values(['severity', 'affected_row'])
issues_df

,affected_row,column,dimension,severity,issue
0,4,customer_id,Completeness,High,Missing or blank value
2,9,amount,Completeness,High,Missing or blank value
6,9,amount,Validity,High,"Invalid amount: non-numeric, non-positive, or ..."
11,21,transaction_date,Validity,High,Invalid or unparsable date
12,22,transaction_date,Validity,High,Invalid or unparsable date
7,31,amount,Validity,High,"Invalid amount: non-numeric, non-positive, or ..."
8,36,amount,Validity,High,"Invalid amount: non-numeric, non-positive, or ..."
9,41,amount,Validity,High,"Invalid amount: non-numeric, non-positive, or ..."
4,50,transaction_id,Uniqueness,High,Duplicate transaction identifier
5,51,transaction_id,Uniqueness,High,Duplicate transaction identifier


In [6]:
total_cells = df.shape[0] * df.shape[1]
completeness_rate = 1 - df.isna().sum().sum() / total_cells
duplication_rate = df.duplicated(subset=['transaction_id']).mean()
amount_validity_rate = ((amount_num.notna()) & (amount_num > 0) & (amount_num <= 10000)).mean()
date_parseability_rate = date_parsed.notna().mean()

kpi = pd.DataFrame({
    'KPI': ['Completeness Rate', 'Duplication Rate', 'Amount Validity Rate', 'Date Parseability Rate'],
    'Value': [completeness_rate, duplication_rate, amount_validity_rate, date_parseability_rate]
})
kpi['Value_%'] = (kpi['Value'] * 100).round(1)
kpi

,KPI,Value,Value_%
0,Completeness Rate,0.997222,99.7
1,Duplication Rate,0.008333,0.8
2,Amount Validity Rate,0.958333,95.8
3,Date Parseability Rate,0.966667,96.7


### KPI interpretation

The completeness rate can look acceptable while still hiding critical missing fields. Duplication and amount validity are usually the most business-sensitive metrics in transaction data.

In [7]:
validation_rules = pd.DataFrame([
    ['R1', 'transaction_id must be non-null and unique', int(df['transaction_id'].isna().sum() + df['transaction_id'].duplicated().sum())],
    ['R2', 'amount must be numeric and between 0 and 10000', int((~amount_num.notna() | (amount_num <= 0) | (amount_num > 10000)).sum())],
    ['R3', 'transaction_date must be parseable', int(date_parsed.isna().sum())],
    ['R4', 'region must be one of EU, US, APAC', int((~df['region'].astype(str).str.upper().isin(['EU', 'US', 'APAC'])).sum())]
], columns=['rule_id', 'validation_rule', 'affected_rows'])
validation_rules

,rule_id,validation_rule,affected_rows
0,R1,transaction_id must be non-null and unique,1
1,R2,amount must be numeric and between 0 and 10000,5
2,R3,transaction_date must be parseable,4
3,R4,"region must be one of EU, US, APAC",0


## Audit summary

This section lists the issue type, affected rows, severity, and the recommended next action.

In [8]:
audit_summary = issues_df.groupby(['dimension', 'severity']).agg(affected_rows=('affected_row', 'count')).reset_index()
audit_summary['recommended_next_action'] = audit_summary['dimension'].map({
    'Completeness': 'Flag or impute missing values depending on business importance.',
    'Uniqueness': 'Remove exact duplicates and enforce a primary-key check.',
    'Validity': 'Standardize formats, coerce invalid values, and review outliers manually.',
    'Consistency': 'Normalize casing and category labels.',
    'Integrity': 'Check dependent fields and date logic.'
})
audit_summary

,dimension,severity,affected_rows,recommended_next_action
0,Completeness,High,3,Flag or impute missing values depending on bus...
1,Completeness,Medium,1,Flag or impute missing values depending on bus...
2,Consistency,Low,29,Normalize casing and category labels.
3,Uniqueness,High,2,Remove exact duplicates and enforce a primary-...
4,Validity,High,9,"Standardize formats, coerce invalid values, an..."


## Cleaning recommendations

Suggested next steps before modeling or reporting: deduplicate on transaction_id, standardize dates to ISO format, normalize categorical text fields, validate amount ranges, and flag missing critical identifiers for review.

In [9]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
issues_df.to_csv(output_dir / 'audit_findings.csv', index=False)
kpi.to_csv(output_dir / 'kpi_summary.csv', index=False)
validation_rules.to_csv(output_dir / 'validation_rules.csv', index=False)
audit_summary.to_csv(output_dir / 'audit_summary.csv', index=False)
print('Saved CSV outputs to output/')

Saved CSV outputs to output/
